# STAT 764 · Lab 7 — One tree, many trees

**Assigned Tue Oct 27 · Due Tue Nov 3, 11:59 pm**

Individual work. Parts 1 and 2 need Meeting 15 (Tuesday Oct 27); Part 3 needs Meeting 16
(Thursday Oct 29), and question **(d)** needs the reading, ISLP §8.2.1–8.2.2.

Same rules as before: `git pull`, copy into `work/` before editing, Restart & Run All before
submitting, upload the `.ipynb` to Canvas.

Class built trees and forests on the readmission data. This lab moves them to the Ames houses
from Meetings 5–8, where you already know what an honest linear model scores. Three questions:
how good is **one** tree, tuned honestly; how much does one tree **change** when the sample
does; and what does **averaging many trees** buy?

⏱ The whole lab runs in about a minute.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import load

In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

# Pool_QC is blank for all but 13 houses; sklearn warns on every fit whose sample has none.
# Harmless, and it would bury your output -- that one message only is switched off.
warnings.filterwarnings("ignore", message="Skipping features without any observed values")

ames = load("ames.csv")
y = ames["SalePrice"]

# Lab 4's WIDE design: every column except the two ID numbers and the answer.
NUMERIC = [c for c in ames.select_dtypes("number").columns
           if c not in ("Order", "PID", "SalePrice")]
CATEGORICAL = ames.select_dtypes(exclude="number").columns.tolist()
X = ames[NUMERIC + CATEGORICAL]


def wide_pipeline(model, scale=True):
    """Lab 4's pipeline on every column. A tree splits on one column at a time, so scaling
    changes nothing for it; scale=False keeps the split thresholds in the columns' own units."""
    numeric_steps = [("fill", SimpleImputer(strategy="median"))]
    if scale:
        numeric_steps.append(("scale", StandardScaler()))
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline(numeric_steps), NUMERIC),
            ("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                     fill_value="Missing")),
                              ("encode", OneHotEncoder(handle_unknown="ignore"))]),
             CATEGORICAL),
        ])),
        ("model", model),
    ])


# Lab 4's split: 733 test houses, everything else to train. Every model below is fitted on
# X_train only and judged once on X_test.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=733, random_state=764)
cv = KFold(5, shuffle=True, random_state=764)       # the folds every search and CV uses


def test_mae(fitted):
    """Held-out mean absolute error of a fitted pipeline, in dollars."""
    return mean_absolute_error(y_test, fitted.predict(X_test))


print(f"train {len(X_train):,} houses · test {len(X_test):,} houses")

Two more helpers for Part 2, and one house to follow through the whole lab:

- `bootstrap_rows(seed)` — a bootstrap sample of the training houses: as many draws as there
  are houses, **with replacement**, so some houses appear twice and about a third not at all.
- `top_splits(fitted)` — the first three questions a fitted tree asks: the split at the top,
  then the split on each side of it, written like `"Overall_Qual <= 7.5"`.

In [ ]:
def bootstrap_rows(seed):
    """Row positions (into X_train) of one bootstrap sample, drawn with replacement."""
    return np.random.default_rng(seed).integers(0, len(X_train), len(X_train))


def top_splits(fitted):
    """(first split, left split, right split) of a fitted wide_pipeline whose model is a tree."""
    tree = fitted.named_steps["model"].tree_
    names = fitted.named_steps["prep"].get_feature_names_out()
    left, right = tree.children_left[0], tree.children_right[0]

    def say(node):
        column = names[tree.feature[node]].split("__", 1)[1]      # drop the "num__" prefix
        return f"{column} <= {tree.threshold[node]:.4g}"
    return say(0), say(left), say(right)


# One house, followed through Parts 2 and 3: the first house in the test set.
house = X_test.iloc[[0]]
print(f"the house: sold for ${y_test.iloc[0]:,.0f} · "
      + " · ".join(f"{c} {house[c].iloc[0]}" for c in
                   ["Overall_Qual", "Gr_Liv_Area", "Year_Built", "Neighborhood"]))

Everything below uses that one split, those folds and fixed seeds, so your numbers will match
mine. Your job is the experiments, not the plumbing.

---

## Part 1 · One tree, tuned honestly (15 points)

Meeting 15's rule: a tree's size is a setting, and settings are tuned **inside**
cross-validation on the training houses — never on the test houses.

**(i)** Tune a tree with `GridSearchCV` over this grid, using `cv` and
`scoring="neg_mean_absolute_error"`:

```python
tree_search = GridSearchCV(
    wide_pipeline(DecisionTreeRegressor(random_state=0), scale=False),
    {"model__max_depth": [2, 3, 4, 6, 8, 12], "model__min_samples_leaf": [1, 5, 20, 50]},
    cv=cv, scoring="neg_mean_absolute_error")
```

Fit it on `X_train, y_train`. Print the cross-validated MAE for **every** depth and leaf size
as one small table — `pd.DataFrame(tree_search.cv_results_)` has the columns, and
`.pivot_table(index="param_model__max_depth", columns="param_model__min_samples_leaf",
values="mean_test_score")` arranges them (the scores are negative; flip the sign).

**(ii)** Tune ridge regression the same way — `wide_pipeline(Ridge())`,
`{"model__alpha": np.logspace(-1, 3, 9)}`, the same `cv` and scoring — Meetings 7–8's model.

Report **one table**: for the tuned tree and the tuned ridge, the settings chosen, the
cross-validated MAE of those settings, and the test MAE (`test_mae`).

In [ ]:
# YOUR CODE HERE

## Part 2 · Fifty trees from fifty samples (15 points)

A tree is built greedily: the best first split, then the best split on each side, and so on.
Meeting 15 called that **unstable** — a slightly different sample can change a split, and
everything below it changes too. Measure it.

For each `seed` in `range(50)`, fit the tree with Part 1's chosen settings
(`**{k.split("__")[1]: v for k, v in tree_search.best_params_.items()}` passes them to
`DecisionTreeRegressor(random_state=0, ...)` inside `wide_pipeline(..., scale=False)`) on the
bootstrap sample `X_train.iloc[bootstrap_rows(seed)]`, `y_train.iloc[bootstrap_rows(seed)]`.
Keep its `top_splits` and its prediction for `house`.

Report:

1. how often each **first** split appears across the 50 trees (`value_counts()`);
2. how many **different** left splits, right splits, and whole top-three structures
   (first, left, right together) the 50 trees use;
3. the house's 50 predictions: median, standard deviation, smallest and largest — beside the
   price it actually sold for.

In [ ]:
# YOUR CODE HERE

## Part 3 · From one tree to a forest (25 points)

Meeting 16's two ideas. **Bagging**: grow many trees on bootstrap samples and average their
predictions. **A random forest**: the same, but each split may only choose among a random
subset of the columns, so the trees differ more and their errors cancel better. Each tree never
sees about a third of the training houses — its **out-of-bag** houses — and predicting each house
with only the trees that did not see it gives an honest error estimate for free:
`oob_prediction_`.

⚠ In scikit-learn, `RandomForestRegressor`'s default is `max_features=1.0` — every column at
every split — which is **bagging**, not a random forest. Set `max_features` yourself.

**(i)** Fit two ensembles of 300 trees on `X_train`, each in `wide_pipeline(..., scale=False)`
with `oob_score=True, random_state=0, n_jobs=-1`:

- **bagging**: `RandomForestRegressor(n_estimators=300, max_features=1.0, ...)`;
- **random forest**: `max_features=1/3` — a third of the columns at each split, a common choice
  for regression.

For each, the test MAE and the out-of-bag MAE:
`mean_absolute_error(y_train, fitted.named_steps["model"].oob_prediction_)`. For the random
forest also the 5-fold cross-validated MAE: `cross_val_score` with the same `cv` and scoring
(fold the sign back), and its smallest and largest fold.

Report **one table** with four rows — the tuned tree and the tuned ridge from Part 1, bagging,
and the random forest — and the columns test MAE, out-of-bag MAE (the ensembles) and
cross-validated MAE (where you have one).

**(ii)** Back to the house. For each `seed` in `range(20)`, fit a random forest of **100**
trees (`max_features=1/3, random_state=seed, n_jobs=-1`) on the bootstrap sample
`bootstrap_rows(seed)`, and keep its prediction for `house`. Report the median, standard
deviation, smallest and largest of the 20 — beside the same four numbers for the **first 20**
of Part 2's trees.

In [ ]:
# YOUR CODE HERE

## Part 4 · Written (20 points)

Three or four sentences each, and **every answer except (d) points at a number in your own
output.** Under each question, *A full answer* says what earns full credit — the same lines
are on the Canvas rubric.

**(a)** From Part 1. Which predicts the test houses better, the tuned tree or ridge, and by
how much? Why can one tree not keep up — what does a tree's prediction look like as a house
gets bigger? And which setting did the tuning lean on to keep the tree from overfitting?

*A full answer:* quotes the two test MAEs and the gap; says a tree predicts one average price
per leaf, so its predictions are a staircase that cannot follow a steady trend such as price
rising with living area; and reads the CV table to say the minimum leaf size (not the depth)
did the restraining, with the numbers for leaf size 1 against the chosen one at that depth.

**(b)** From Part 2. What stayed the same across the fifty trees, and what changed? What
does that mean for reading one tree's splits as "the factors that matter"? And for the house?

*A full answer:* quotes how often the most common first split appears, and the counts of
different left, right and top-three structures; says one tree's story below the top is largely
an accident of its sample; and quotes the house's range and standard deviation against its
actual price — one tree's prediction for one house depends heavily on which houses it saw.

**(c)** From Part 3. Rank the four models on the test houses. How close are the out-of-bag
and cross-validated estimates to the test MAE, and why is the out-of-bag estimate honest even
though no houses were set aside? How much did averaging shrink the spread of the house's
prediction, and what did using a third of the columns add over bagging?

*A full answer:* quotes the four test MAEs in order; quotes the forest's out-of-bag and
cross-validated MAEs beside its test MAE and says each house is predicted only by trees that
never saw it; quotes the two standard deviations for the house (forests against trees) and
says averaging cancels the trees' separate errors; and quotes bagging against the random
forest, explaining the gain through less similar trees — Part 2's first split shows why bagged
trees are alike: one strong column wins the top of almost every tree.

**(d)** 📗 **From the reading.** ISLP §8.2.1–8.2.2. Why does averaging many bootstrapped
trees reduce variance — what does the book's argument about averaging *n* observations say?
Why does that gain shrink when the trees are alike, and when are they alike? How does a random
forest change that, and what value of *m* does the book suggest? About what share of the
observations is out of bag for each tree?

*A full answer:* averaging *n* independent quantities with variance σ² gives variance σ²/*n*;
averaging highly correlated trees reduces variance much less, and bagged trees are alike when
one strong predictor wins the top split in most of them; a random forest considers a random
subset of *m* predictors at each split (the book suggests *m* ≈ √*p*), which decorrelates the
trees; about a third of the observations are out of bag for each tree. No number from your own
output needed.

⚠ Answer (d) from the reading. It is there, and it is short.

### Your answers

*(double-click to edit)*

**(a)**

**(b)**

**(c)**

**(d)**

---

## Before you submit

- [ ] **Run → Restart Kernel and Run All Cells**
- [ ] Part 1 has the CV table for every depth and leaf size, and one table for the tuned tree
      and ridge: settings, CV MAE, test MAE
- [ ] Part 2 has the first-split counts, the numbers of different splits and structures, and
      the house's median, standard deviation, smallest and largest prediction
- [ ] Part 3 has the four-row table (test, out-of-bag and CV MAE) and the house's spread for
      20 forests beside 20 trees
- [ ] All four written answers are filled in, (a)–(c) each pointing at a number
- [ ] Uploaded the `.ipynb` to Canvas

**75 points.**